# Prompt Laboratory — Interactive Notebook

A companion notebook to the **LLM Playground & Prompt Laboratory** app.

This notebook walks through the same core mechanics as the Streamlit app, but in an exploratory,
cell-by-cell format:

1. Setup & authentication
2. Prompt experimentation
3. Temperature comparison
4. Token counting
5. Cost estimation
6. Latency measurement
7. Saving prompt history

> Runs in Google Colab or any local Jupyter environment with Python 3.9+.

## 1. Setup

In [ ]:
%pip install -q openai tiktoken pandas matplotlib


In [ ]:
import getpass
import os

# Prompted securely so the key never lands in notebook output or version control.
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")


In [ ]:
import json
import time
from datetime import datetime, timezone

import pandas as pd
import tiktoken
from openai import OpenAI

client = OpenAI()
MODEL = "gpt-4o-mini"


## 2. Prompt experimentation

A small helper wraps the raw API call so the rest of the notebook can call one function
instead of repeating boilerplate.

In [ ]:
def run_prompt(prompt: str, temperature: float = 0.7, top_p: float = 1.0, max_tokens: int = 300):
    """Send a single prompt to the model and return the response plus timing/usage."""
    start = time.perf_counter()
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens,
    )
    elapsed = round(time.perf_counter() - start, 3)
    return {
        "text": response.choices[0].message.content,
        "input_tokens": response.usage.prompt_tokens,
        "output_tokens": response.usage.completion_tokens,
        "latency_seconds": elapsed,
    }


result = run_prompt("Explain what a REST API is in two sentences.")
print(result["text"])
result


## 3. Temperature comparison

Run the same prompt at several temperature values to see how output variability changes.

In [ ]:
prompt = "Write a one-line tagline for a productivity app."
temperatures = [0.0, 0.5, 1.0, 1.5]

comparison_rows = []
for temp in temperatures:
    outcome = run_prompt(prompt, temperature=temp)
    comparison_rows.append({"temperature": temp, **outcome})

comparison_df = pd.DataFrame(comparison_rows)
comparison_df


## 4. Token counting

`tiktoken` gives an exact token count for a given model's tokenizer — useful for
estimating cost *before* making an API call.

In [ ]:
def count_tokens(text: str, model: str = MODEL) -> int:
    try:
        encoding = tiktoken.encoding_for_model(model)
    except KeyError:
        encoding = tiktoken.get_encoding("cl100k_base")
    return len(encoding.encode(text))


sample_text = "Large language models tokenize text before processing it."
print(f"Token count: {count_tokens(sample_text)}")


## 5. Cost estimation

A minimal price table (USD per 1K tokens) mirroring `utils/pricing.py` in the main app.

In [ ]:
PRICE_PER_1K = {
    "gpt-4o": {"input": 0.0050, "output": 0.0150},
    "gpt-4o-mini": {"input": 0.00015, "output": 0.00060},
}


def estimate_cost(model: str, input_tokens: int, output_tokens: int) -> float:
    price = PRICE_PER_1K.get(model, {"input": 0.0, "output": 0.0})
    return round(
        (input_tokens / 1000) * price["input"] + (output_tokens / 1000) * price["output"], 6
    )


comparison_df["estimated_cost_usd"] = comparison_df.apply(
    lambda row: estimate_cost(MODEL, row["input_tokens"], row["output_tokens"]), axis=1
)
comparison_df


## 6. Latency measurement

Visualize how latency varies across the temperature sweep run earlier.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.bar(comparison_df["temperature"].astype(str), comparison_df["latency_seconds"])
plt.xlabel("Temperature")
plt.ylabel("Latency (seconds)")
plt.title("Response Latency by Temperature")
plt.tight_layout()
plt.show()


## 7. Saving prompt history

Persist the experiment results to a local JSON file, matching the schema used by the
Streamlit app's `utils/storage.py` so files are interchangeable between the two.

In [ ]:
history_path = "notebook_prompt_history.json"
records = []

for _, row in comparison_df.iterrows():
    records.append(
        {
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "model": MODEL,
            "temperature": row["temperature"],
            "prompt": prompt,
            "response": row["text"],
            "input_tokens": int(row["input_tokens"]),
            "output_tokens": int(row["output_tokens"]),
            "total_tokens": int(row["input_tokens"] + row["output_tokens"]),
            "estimated_cost": row["estimated_cost_usd"],
            "latency_seconds": row["latency_seconds"],
        }
    )

with open(history_path, "w", encoding="utf-8") as f:
    json.dump(records, f, indent=2)

print(f"Saved {len(records)} records to {history_path}")


## Next steps

- Point this notebook at the same `prompts/prompt_history.json` used by the Streamlit app to keep one unified history.
- Extend `PRICE_PER_1K` as new models are added.
- Swap `run_prompt` for a Gemini/Anthropic call to compare providers side by side.